In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
# Load the datasets
train_df = pd.read_csv("../Data/train_data.csv")
val_df = pd.read_csv("../Data/val_data.csv")
test_df = pd.read_csv("../Data/test_data.csv")

In [ ]:
# Create country-level average feature
country_avg = train_df.groupby(["country_name", "series_name"])["value"].mean().reset_index()
country_avg.columns = ["country_name", "series_name", "country_avg_value"]
train_df = train_df.merge(country_avg, on=["country_name", "series_name"], how="left")
val_df = val_df.merge(country_avg, on=["country_name", "series_name"], how="left")
test_df = test_df.merge(country_avg, on=["country_name", "series_name"], how="left")

In [4]:
#  Pivot 'value' and 'country_avg_value'
value_wide_train = train_df.pivot_table(index=["country_name", "year"], columns="series_name", values="value")
value_wide_val = val_df.pivot_table(index=["country_name", "year"], columns="series_name", values="value")
value_wide_test = test_df.pivot_table(index=["country_name", "year"], columns="series_name", values="value")

avg_wide_train = train_df.pivot_table(index=["country_name", "year"], columns="series_name", values="country_avg_value")
avg_wide_val = val_df.pivot_table(index=["country_name", "year"], columns="series_name", values="country_avg_value")
avg_wide_test = test_df.pivot_table(index=["country_name", "year"], columns="series_name", values="country_avg_value")
avg_wide_train.columns = ["avg_" + col for col in avg_wide_train.columns]
avg_wide_val.columns = ["avg_" + col for col in avg_wide_val.columns]
avg_wide_test.columns = ["avg_" + col for col in avg_wide_test.columns]

train_wide = pd.concat([value_wide_train, avg_wide_train], axis=1).reset_index()
val_wide = pd.concat([value_wide_val, avg_wide_val], axis=1).reset_index()
test_wide = pd.concat([value_wide_test, avg_wide_test], axis=1).reset_index()

In [5]:
# Data-centric AI #1: country-level averages
# We computed long-term averages for each indicator by country using the training set. 
# This provides a baseline for each country, and helps the model distinguish whether a given year's value is high or low relative to historical norms.
group_avg = train_df.groupby(["income_group", "year", "series_name"])["value"].mean().reset_index()
group_avg.columns = ["income_group", "year", "series_name", "income_group_avg_value"]
train_df = train_df.merge(group_avg, on=["income_group", "year", "series_name"], how="left")
val_df = val_df.merge(group_avg, on=["income_group", "year", "series_name"], how="left")
test_df = test_df.merge(group_avg, on=["income_group", "year", "series_name"], how="left")

groupavg_train = train_df.pivot_table(index=["country_name", "year"], columns="series_name", values="income_group_avg_value")
groupavg_val = val_df.pivot_table(index=["country_name", "year"], columns="series_name", values="income_group_avg_value")
groupavg_test = test_df.pivot_table(index=["country_name", "year"], columns="series_name", values="income_group_avg_value")
groupavg_train.columns = ["groupavg_" + col for col in groupavg_train.columns]
groupavg_val.columns = ["groupavg_" + col for col in groupavg_val.columns]
groupavg_test.columns = ["groupavg_" + col for col in groupavg_test.columns]

train_wide = pd.concat([train_wide, groupavg_train.reset_index(drop=True)], axis=1)
val_wide = pd.concat([val_wide, groupavg_val.reset_index(drop=True)], axis=1)
test_wide = pd.concat([test_wide, groupavg_test.reset_index(drop=True)], axis=1)


In [6]:
# Data-centric AI #2: income group-year averages
# We calculated average indicator values grouped by income level and year.
# This captures global structural patterns tied to economic tiers, which gives the model more context to compare a country’s standing relative to other countries within the same income group.

# Remove column to avoid merge conflicts
for df in [train_df, val_df, test_df]:
    if "income_group_avg_value" in df.columns:
        df.drop(columns=["income_group_avg_value"], inplace=True)

# Create group-year averages from training
group_avg = train_df.groupby(["income_group", "year", "series_name"])["value"].mean().reset_index()
group_avg.columns = ["income_group", "year", "series_name", "income_group_avg_value"]

# Merge into train, validation, and test sets
train_df = train_df.merge(group_avg, on=["income_group", "year", "series_name"], how="left")
val_df = val_df.merge(group_avg, on=["income_group", "year", "series_name"], how="left")
test_df = test_df.merge(group_avg, on=["income_group", "year", "series_name"], how="left")

# Pivot into wide format
groupavg_train = train_df.pivot_table(index=["country_name", "year"], columns="series_name", values="income_group_avg_value")
groupavg_val = val_df.pivot_table(index=["country_name", "year"], columns="series_name", values="income_group_avg_value")
groupavg_test = test_df.pivot_table(index=["country_name", "year"], columns="series_name", values="income_group_avg_value")

# Rename columns
groupavg_train.columns = ["groupavg_" + col for col in groupavg_train.columns]
groupavg_val.columns = ["groupavg_" + col for col in groupavg_val.columns]
groupavg_test.columns = ["groupavg_" + col for col in groupavg_test.columns]

# Merge into final datasets
train_wide = pd.concat([train_wide, groupavg_train.reset_index(drop=True)], axis=1)
val_wide = pd.concat([val_wide, groupavg_val.reset_index(drop=True)], axis=1)
test_wide = pd.concat([test_wide, groupavg_test.reset_index(drop=True)], axis=1)


In [7]:
# Data-centric AI #3: temporal delta features
# We calculated year over year change in indicator values per country and indicator.
# This helps the model understand whether indicators are improving, declining, or stable over time.

# Sort data
train_df = train_df.sort_values(by=["country_name", "series_name", "year"])
val_df = val_df.sort_values(by=["country_name", "series_name", "year"])
test_df = test_df.sort_values(by=["country_name", "series_name", "year"])

# Compute delta
train_df["delta_value"] = train_df.groupby(["country_name", "series_name"])["value"].transform(lambda x: x.diff())
val_df["delta_value"] = val_df.groupby(["country_name", "series_name"])["value"].transform(lambda x: x.diff())
test_df["delta_value"] = test_df.groupby(["country_name", "series_name"])["value"].transform(lambda x: x.diff())

# Fill NaNs with 0
train_df["delta_value"] = train_df["delta_value"].fillna(0)
val_df["delta_value"] = val_df["delta_value"].fillna(0)
test_df["delta_value"] = test_df["delta_value"].fillna(0)

# Pivot into wide format
delta_wide_train = train_df.pivot_table(index=["country_name", "year"], columns="series_name", values="delta_value")
delta_wide_val = val_df.pivot_table(index=["country_name", "year"], columns="series_name", values="delta_value")
delta_wide_test = test_df.pivot_table(index=["country_name", "year"], columns="series_name", values="delta_value")

# Rename delta columns
delta_wide_train.columns = ["delta_" + col for col in delta_wide_train.columns]
delta_wide_val.columns = ["delta_" + col for col in delta_wide_val.columns]
delta_wide_test.columns = ["delta_" + col for col in delta_wide_test.columns]

# Merge into wide datasets
train_wide = pd.concat([train_wide, delta_wide_train.reset_index(drop=True)], axis=1)
val_wide = pd.concat([val_wide, delta_wide_val.reset_index(drop=True)], axis=1)
test_wide = pd.concat([test_wide, delta_wide_test.reset_index(drop=True)], axis=1)


In [9]:
# Clean columns
bad_cols = [col for col in train_wide.columns if "Last Updated" in col or "Data from database" in col]
train_wide = train_wide.drop(columns=[col for col in bad_cols if col in train_wide.columns])
val_wide = val_wide.drop(columns=[col for col in bad_cols if col in val_wide.columns])
test_wide = test_wide.drop(columns=[col for col in bad_cols if col in test_wide.columns])

In [10]:
# Prepare x and y
target_col = "Life expectancy at birth, total (years)"
drop_cols = ["country_name", "year", target_col]
X_train = train_wide.drop(columns=drop_cols)
y_train = train_wide[target_col]
X_val = val_wide.drop(columns=drop_cols)
y_val = val_wide[target_col]
X_test = test_wide.drop(columns=drop_cols)
y_test = test_wide[target_col]

# Align columns
common_cols = list(set(X_train.columns) & set(X_val.columns) & set(X_test.columns))
X_train = X_train[common_cols]
X_val = X_val[common_cols]
X_test = X_test[common_cols]

In [11]:
# Train model
xgb_final = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    colsample_bytree=1.0,
    subsample=1.0,
    random_state=42
)
xgb_final.fit(X_train, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=1.0, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.1, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=100, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [ ]:
# Evaluate
def evaluate_model(model, X, y, name):
    preds = model.predict(X)
    rmse = np.sqrt(mean_squared_error(y, preds))
    r2 = r2_score(y, preds)
    print(f"{name} RMSE: {rmse:.4f}")
    print(f"{name} R²: {r2:.4f}")
    return rmse, r2

print("\nModel evaluation after data improvements")
train_rmse, train_r2 = evaluate_model(xgb_final, X_train, y_train, "Train")
val_rmse, val_r2 = evaluate_model(xgb_final, X_val, y_val, "Validation")
test_rmse, test_r2 = evaluate_model(xgb_final, X_test, y_test, "Test")


Model evaluation after data improvements
Train RMSE: 0.0263
Train R²: 1.0000
Validation RMSE: 1.1244
Validation R²: 0.9812
Test RMSE: 1.9312
Test R²: 0.9442


In [13]:
# Summary
summary = pd.DataFrame({
    "Dataset": ["Train", "Validation", "Test"],
    "RMSE": [train_rmse, val_rmse, test_rmse],
    "R²": [train_r2, val_r2, test_r2]
})
print("\nFinal model performance summary")
print(summary)


Final model performance summary
      Dataset      RMSE        R²
0       Train  0.026317  0.999993
1  Validation  1.124350  0.981193
2        Test  1.931236  0.944196
